# BUSI End-to-End: Attention U-Net + Lesion-Mask-Guided EfficientNet

**Pipeline (single dataset = BUSI, single notebook):**

```
Stage A:  Train Attention U-Net on BUSI train split (TF/Keras)
Stage B:  Run U-Net on all splits → save soft pred_mask + image + label as .npz
Stage C:  Train EfficientNet-B0 with soft-guided attention (PyTorch)
          using U-Net's predicted mask as the attention prior
```

**Task:** binary classification — `malignant` vs. `benign+normal`.

**Why this works:** U-Net learns *where* the lesion is. The classifier learns
*what* the lesion looks like, with attention biased toward U-Net's predicted
region. Because U-Net is trained only on the train split and we use its
**predictions** (not GT) on every split, the classifier sees a consistent mask
distribution at train and test time — no leakage.

**Skip flags** at the top of the notebook let you re-run later sections without
re-training the U-Net.


---
## 0. Setup

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install (quietly). Skip if already done.
from IPython.display import clear_output
!pip install -q opencv-python-headless scikit-learn
clear_output()
print("Packages ready.")

In [ ]:
# === Paths & global config ===
import os, json, random, gc
import numpy as np

# --- Dataset & output paths ---
DRIVE_ROOT     = '/content/drive/MyDrive/breast cancer- segment'
BUSI_DIR       = os.path.join(DRIVE_ROOT, 'Dataset_BUSI_with_GT')

INTEG_DIR      = os.path.join(DRIVE_ROOT, 'integrated_v1')
UNET_WEIGHTS   = os.path.join(INTEG_DIR, 'unet_busi.weights.h5')
SPLIT_PATH     = os.path.join(INTEG_DIR, 'split_manifest.json')
PROCESSED_DIR  = os.path.join(INTEG_DIR, 'processed')   # .npz cache for Stage C
CLF_WEIGHTS    = os.path.join(INTEG_DIR, 'classifier.pth')
os.makedirs(INTEG_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# --- Hyperparams (shared) ---
SIZE        = 256          # image size everywhere (U-Net + classifier)
SEED        = 42

# --- Stage skip flags ---
# Set to True to skip a stage (will load cached artifacts instead).
SKIP_UNET_TRAINING = False     # Skip if UNET_WEIGHTS already on Drive
SKIP_BRIDGE        = False     # Skip if PROCESSED_DIR already populated

# --- Class mapping ---
# BUSI folders -> 3-class index (alphabetical) -> binary task
THREE_CLASS = {'benign': 0, 'malignant': 1, 'normal': 2}
# Binary task: malignant vs non-malignant
TO_BINARY   = {0: 0, 1: 1, 2: 0}   # benign->0, malignant->1, normal->0
BINARY_NAMES = ['Non-malignant', 'Malignant']

# --- Reproducibility ---
random.seed(SEED); np.random.seed(SEED)

assert os.path.exists(BUSI_DIR), f'BUSI dataset not found at {BUSI_DIR}'
print('BUSI dir       :', BUSI_DIR)
print('Integration dir:', INTEG_DIR)

---
## 1. Data Preparation — stratified train/val/test split

We list every `(image, mask, class)` tuple in BUSI and split 60/20/20,
stratified by 3-class label. The split is saved to Drive so re-running the
notebook reuses exactly the same partition.

> Multi-mask samples (`*_mask_1.png`) are merged with the primary mask via
> logical OR so the GT covers all lesion regions.


In [ ]:
from glob import glob

def collect_busi(busi_dir):
    """Return list of dicts: {image_path, mask_paths(list), cls(0/1/2), stem}."""
    items = []
    for cls_name, cls_idx in THREE_CLASS.items():
        cls_dir = os.path.join(busi_dir, cls_name)
        # primary masks: '*_mask.png'
        primaries = sorted(glob(os.path.join(cls_dir, '*_mask.png')))
        for mp in primaries:
            stem = os.path.basename(mp).replace('_mask.png', '')
            ip = os.path.join(cls_dir, stem + '.png')
            if not os.path.exists(ip):
                continue
            # extra masks (multi-region lesions)
            extras = sorted(glob(os.path.join(cls_dir, f'{stem}_mask_*.png')))
            items.append({
                'image_path': ip,
                'mask_paths': [mp] + extras,
                'cls': cls_idx,
                'stem': f'{cls_name}_{stem}',
            })
    return items

all_items = collect_busi(BUSI_DIR)
counts = {k: sum(1 for x in all_items if x['cls'] == v) for k, v in THREE_CLASS.items()}
print(f"Total samples: {len(all_items)}")
print(f"Per-class    : {counts}")
print(f"Multi-mask samples: {sum(1 for x in all_items if len(x['mask_paths']) > 1)}")

In [ ]:
from sklearn.model_selection import train_test_split

if os.path.exists(SPLIT_PATH):
    with open(SPLIT_PATH) as f:
        manifest = json.load(f)
    print(f'Loaded existing split from {SPLIT_PATH}')
else:
    labels = [x['cls'] for x in all_items]
    train_items, temp_items, _, temp_lbl = train_test_split(
        all_items, labels, test_size=0.4, stratify=labels, random_state=SEED)
    val_items, test_items = train_test_split(
        temp_items, test_size=0.5, stratify=temp_lbl, random_state=SEED)

    manifest = {
        'train': [x['stem'] for x in train_items],
        'val'  : [x['stem'] for x in val_items],
        'test' : [x['stem'] for x in test_items],
    }
    with open(SPLIT_PATH, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f'Saved split to {SPLIT_PATH}')

# Build name->item lookup, then split lists
by_stem = {x['stem']: x for x in all_items}
train_items = [by_stem[s] for s in manifest['train']]
val_items   = [by_stem[s] for s in manifest['val']]
test_items  = [by_stem[s] for s in manifest['test']]

def show_split(name, items):
    cs = {0: 0, 1: 0, 2: 0}
    for x in items: cs[x['cls']] += 1
    print(f'{name:6s} n={len(items):4d} | benign={cs[0]:3d} malignant={cs[1]:3d} normal={cs[2]:3d}')
for n, items in [('train', train_items), ('val', val_items), ('test', test_items)]:
    show_split(n, items)

---
## 2. Stage A — Train Attention U-Net (TF/Keras)

Differences from the original segmentation notebook:

- 1-channel input (256×256×1) — BUSI is grayscale, no RGB upcast.
- Stratified split (re-uses `split_manifest.json`).
- Masks explicitly binarized; multi-mask samples merged.
- Loss = BCE + Dice (handles small-lesion class imbalance).
- Light, anatomy-respecting augmentation (flips + intensity).
- Metric tracked for best-model selection: per-image Dice (manual eval each epoch).


In [ ]:
# TensorFlow imports — only used in Stage A and Stage B
import tensorflow as tf
import cv2
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Layer, Conv2D, Dropout, UpSampling2D,
                                      MaxPool2D, BatchNormalization,
                                      Add, Multiply, concatenate)
from tensorflow.keras.callbacks import Callback, ModelCheckpoint
tf.keras.utils.set_random_seed(SEED)

print('TF version:', tf.__version__)
print('GPU      :', tf.config.list_physical_devices('GPU'))

In [ ]:
# --- Attention U-Net building blocks (1-channel friendly) ---
class EncoderBlock(Layer):
    def __init__(self, filters, rate, pooling=True, **kw):
        super().__init__(**kw)
        self.filters, self.rate, self.pooling = filters, rate, pooling
        self.c1 = Conv2D(filters, 3, padding='same', activation='relu', kernel_initializer='he_normal')
        self.dr = Dropout(rate)
        self.c2 = Conv2D(filters, 3, padding='same', activation='relu', kernel_initializer='he_normal')
        self.mp = MaxPool2D()
    def call(self, x):
        x = self.c2(self.dr(self.c1(x)))
        return (self.mp(x), x) if self.pooling else x
    def get_config(self):
        return {**super().get_config(),
                'filters': self.filters, 'rate': self.rate, 'pooling': self.pooling}

class DecoderBlock(Layer):
    def __init__(self, filters, rate, **kw):
        super().__init__(**kw)
        self.filters, self.rate = filters, rate
        self.up  = UpSampling2D()
        self.net = EncoderBlock(filters, rate, pooling=False)
    def call(self, x):
        x, skip = x
        x = self.up(x)
        return self.net(concatenate([x, skip]))
    def get_config(self):
        return {**super().get_config(), 'filters': self.filters, 'rate': self.rate}

class AttentionGate(Layer):
    def __init__(self, filters, bn=True, **kw):
        super().__init__(**kw)
        self.filters, self.bn = filters, bn
        self.normal = Conv2D(filters, 3, padding='same', activation='relu', kernel_initializer='he_normal')
        self.down   = Conv2D(filters, 3, strides=2, padding='same', activation='relu', kernel_initializer='he_normal')
        self.learn  = Conv2D(1, 1, padding='same', activation='sigmoid')
        self.up     = UpSampling2D()
        self.bn_l   = BatchNormalization()
    def call(self, x):
        gate, skip = x
        a = self.learn(Add()([self.normal(gate), self.down(skip)]))
        a = self.up(a)
        f = Multiply()([a, skip])
        return self.bn_l(f) if self.bn else f
    def get_config(self):
        return {**super().get_config(), 'filters': self.filters, 'bn': self.bn}

In [ ]:
# --- Build Attention U-Net (1-channel input) ---
def build_unet(size=SIZE):
    inp = Input(shape=(size, size, 1))
    p1, c1 = EncoderBlock(32, 0.1, name='Enc1')(inp)
    p2, c2 = EncoderBlock(64, 0.1, name='Enc2')(p1)
    p3, c3 = EncoderBlock(128, 0.2, name='Enc3')(p2)
    p4, c4 = EncoderBlock(256, 0.2, name='Enc4')(p3)
    enc    = EncoderBlock(512, 0.3, pooling=False, name='Bottleneck')(p4)

    a1 = AttentionGate(256, name='Att1')([enc, c4])
    d1 = DecoderBlock(256, 0.2, name='Dec1')([enc, a1])
    a2 = AttentionGate(128, name='Att2')([d1, c3])
    d2 = DecoderBlock(128, 0.2, name='Dec2')([d1, a2])
    a3 = AttentionGate(64,  name='Att3')([d2, c2])
    d3 = DecoderBlock(64, 0.1, name='Dec3')([d2, a3])
    a4 = AttentionGate(32,  name='Att4')([d3, c1])
    d4 = DecoderBlock(32, 0.1, name='Dec4')([d3, a4])
    out = Conv2D(1, 1, activation='sigmoid', padding='same')(d4)
    return Model(inp, out, name='AttentionUNet')

# --- Loss & metrics ---
def dice_coef(y_true, y_pred, smooth=1.0):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    inter  = tf.reduce_sum(y_true * y_pred)
    return (2.*inter + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def dice_loss(y_true, y_pred, smooth=1.0):
    inter = tf.reduce_sum(y_true * y_pred)
    return 1 - (2.*inter + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def bce_dice(y_true, y_pred):
    return tf.keras.losses.binary_crossentropy(y_true, y_pred) + dice_loss(y_true, y_pred)

In [ ]:
# --- Load images & masks into numpy arrays for given items ---
def load_gray(path, size=SIZE):
    im = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if im is None:
        raise IOError(path)
    im = cv2.resize(im, (size, size), interpolation=cv2.INTER_LINEAR)
    return (im.astype(np.float32) / 255.)[..., None]

def load_mask_union(mask_paths, size=SIZE):
    """Union all mask files into one binary mask."""
    out = np.zeros((size, size), dtype=np.uint8)
    for p in mask_paths:
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        m = cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST)
        out |= (m > 127).astype(np.uint8)
    return out[..., None].astype(np.float32)

def items_to_arrays(items):
    n = len(items)
    X = np.zeros((n, SIZE, SIZE, 1), dtype=np.float32)
    Y = np.zeros((n, SIZE, SIZE, 1), dtype=np.float32)
    for i, x in enumerate(items):
        X[i] = load_gray(x['image_path'])
        Y[i] = load_mask_union(x['mask_paths'])
    return X, Y

if not SKIP_UNET_TRAINING:
    print('Loading train+val into memory...')
    X_tr, Y_tr = items_to_arrays(train_items)
    X_va, Y_va = items_to_arrays(val_items)
    print(f'Train: {X_tr.shape}  | Val: {X_va.shape}')
    print(f'Mean lesion fraction (train): {Y_tr.mean():.4f}')

In [ ]:
# --- tf.data pipeline with augmentation ---
def augment(img, mask):
    if tf.random.uniform([]) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
    if tf.random.uniform([]) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
    img = tf.image.random_brightness(img, 0.05)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    img = tf.clip_by_value(img, 0., 1.)
    return img, mask

BATCH_UNET = 8
EPOCHS_UNET = 25

if not SKIP_UNET_TRAINING:
    train_ds = (tf.data.Dataset.from_tensor_slices((X_tr, Y_tr))
                .shuffle(len(X_tr), seed=SEED)
                .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
                .batch(BATCH_UNET).prefetch(tf.data.AUTOTUNE))
    val_ds   = (tf.data.Dataset.from_tensor_slices((X_va, Y_va))
                .batch(BATCH_UNET).prefetch(tf.data.AUTOTUNE))

In [ ]:
# --- Train (or load) U-Net ---
unet = build_unet(SIZE)

if SKIP_UNET_TRAINING and os.path.exists(UNET_WEIGHTS):
    unet.load_weights(UNET_WEIGHTS)
    print(f'Loaded existing U-Net weights from {UNET_WEIGHTS}')
else:
    unet.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=bce_dice,
        metrics=[dice_coef],
    )
    cbs = [
        ModelCheckpoint(UNET_WEIGHTS, monitor='val_dice_coef',
                        mode='max', save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_dice_coef', mode='max', factor=0.5, patience=4, min_lr=1e-6),
    ]
    history = unet.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_UNET, callbacks=cbs)

    # restore best
    unet.load_weights(UNET_WEIGHTS)
    print(f'Best val dice: {max(history.history["val_dice_coef"]):.4f}')

In [ ]:
# --- Quick U-Net evaluation on test set (sanity check) ---
def per_image_dice(true, pred_bin):
    out = []
    for t, p in zip(true, pred_bin):
        t, p = t.flatten().astype(np.uint8), p.flatten().astype(np.uint8)
        if t.sum() == 0 and p.sum() == 0:
            out.append(1.0); continue
        if t.sum() == 0 or p.sum() == 0:
            out.append(0.0); continue
        out.append(2*(t & p).sum() / (t.sum() + p.sum()))
    return np.array(out)

X_te, Y_te = items_to_arrays(test_items)
P_te = unet.predict(X_te, batch_size=8, verbose=1)
P_te_bin = (P_te > 0.5).astype(np.uint8)

dices = per_image_dice(Y_te, P_te_bin)
print(f'\nTest set (n={len(X_te)}): mean Dice = {dices.mean():.4f}, median = {np.median(dices):.4f}')
print(f'Fraction Dice>0.5 = {(dices>0.5).mean()*100:.1f}%, Dice>0.7 = {(dices>0.7).mean()*100:.1f}%')

# Per-class breakdown — sanity check normal class predicts ~empty masks
for cls_idx, cls_name in [(0, 'benign'), (1, 'malignant'), (2, 'normal')]:
    idxs = [i for i, x in enumerate(test_items) if x['cls'] == cls_idx]
    if idxs:
        d = dices[idxs]
        print(f'  {cls_name:10s} n={len(idxs):3d}  mean Dice = {d.mean():.4f}')

In [ ]:
# --- Visualize a few test predictions ---
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
rng = np.random.RandomState(0)
sample_idx = rng.choice(len(X_te), 4, replace=False)
for r, i in enumerate(sample_idx):
    cls = ['benign', 'malignant', 'normal'][test_items[i]['cls']]
    axes[r,0].imshow(X_te[i,...,0], cmap='gray');     axes[r,0].set_title(f'Image ({cls})')
    axes[r,1].imshow(Y_te[i,...,0], cmap='copper');   axes[r,1].set_title('GT mask')
    axes[r,2].imshow(P_te[i,...,0], cmap='hot', vmin=0, vmax=1); axes[r,2].set_title('Pred (soft)')
    axes[r,3].imshow(P_te_bin[i,...,0], cmap='copper'); axes[r,3].set_title(f'Pred>0.5 (Dice={dices[i]:.2f})')
    for a in axes[r]: a.axis('off')
plt.tight_layout(); plt.show()

---
## 3. Stage B — Generate predicted masks for all splits

Run the trained U-Net on every BUSI image (train + val + test) and save:
- `image`     : float32 (256, 256), grayscale [0,1]
- `pred_mask` : float32 (256, 256), **soft** sigmoid output [0,1] ← what classifier sees
- `gt_mask`   : uint8   (256, 256), binary GT (for ablation only)
- `cls`       : int     ∈ {0,1,2} (benign/malignant/normal)

Storage: `{PROCESSED_DIR}/{train|val|test}/{cls}/{stem}.npz`


In [ ]:
from tqdm.auto import tqdm

def stage_b_run(items, split_name):
    out_root = os.path.join(PROCESSED_DIR, split_name)
    n_done = 0
    for x in tqdm(items, desc=f'Stage B [{split_name}]'):
        cls_dir = os.path.join(out_root, str(x['cls']))
        os.makedirs(cls_dir, exist_ok=True)
        out_path = os.path.join(cls_dir, x['stem'] + '.npz')
        if SKIP_BRIDGE and os.path.exists(out_path):
            n_done += 1; continue

        img  = load_gray(x['image_path'])           # (H,W,1) float32
        gt   = load_mask_union(x['mask_paths'])     # (H,W,1) float32 in {0,1}
        pred = unet.predict(img[None], verbose=0)[0]  # (H,W,1) float32 in [0,1]

        np.savez_compressed(
            out_path,
            image=img.squeeze().astype(np.float32),
            pred_mask=pred.squeeze().astype(np.float32),     # SOFT
            gt_mask=gt.squeeze().astype(np.uint8),
            cls=np.int32(x['cls']),
        )
        n_done += 1
    print(f'  {split_name}: {n_done} files written/verified')

if not (SKIP_BRIDGE and len(os.listdir(PROCESSED_DIR)) > 0):
    for name, items in [('train', train_items), ('val', val_items), ('test', test_items)]:
        stage_b_run(items, name)
else:
    print(f'SKIP_BRIDGE=True and {PROCESSED_DIR} already populated.')

In [ ]:
# Free TF / GPU memory before PyTorch starts
del unet
if not SKIP_UNET_TRAINING:
    del X_tr, Y_tr, X_va, Y_va
del X_te, Y_te, P_te, P_te_bin
tf.keras.backend.clear_session()
gc.collect()
print('TF resources freed.')

---
## 4. Stage C — Lesion-Mask-Guided Classifier (PyTorch)

EfficientNet-B0 (1-channel, ImageNet-pretrained first conv averaged), with a
**Soft-Guided Spatial Attention** module that takes U-Net's soft mask as a
prior. Binary task: malignant vs. non-malignant.

Improvements over the original integrated_v3 notebook:
- Anatomy-respecting augmentation only (flip + small intensity jitter).
- Mixed-precision training (`torch.amp`).
- `WeightedRandomSampler` for class balance (more stable than CE-weights at small batch).
- BatchNorm running stats frozen during fine-tuning (BS=4 is too small for stable BN).
- Best-model selection by **macro-F1**, not accuracy.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score, accuracy_score)

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
class BUSILesionDataset(Dataset):
    """
    Reads .npz files written by Stage B.
    Returns (image, mask, label) where:
        image: (1, H, W) float32 in [0,1]
        mask : (1, H, W) float32 in [0,1]   (soft mask if mask_source='pred')
        label: int (binary, via TO_BINARY)
    """
    def __init__(self, split_dir, augment=False, mask_source='pred'):
        assert mask_source in ('pred', 'gt')
        self.augment = augment
        self.mask_source = mask_source
        self.files = []
        for cls_idx in [0, 1, 2]:
            cls_dir = os.path.join(split_dir, str(cls_idx))
            if os.path.isdir(cls_dir):
                self.files.extend(sorted(glob(os.path.join(cls_dir, '*.npz'))))
        # cache binary labels for the sampler
        self.binary_labels = []
        for f in self.files:
            cls = int(os.path.basename(os.path.dirname(f)))
            self.binary_labels.append(TO_BINARY[cls])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        img  = d['image'].astype(np.float32)
        mask = (d['pred_mask'] if self.mask_source == 'pred'
                else d['gt_mask'].astype(np.float32))
        cls_3 = int(d['cls'])
        label = TO_BINARY[cls_3]

        if self.augment:
            # h-flip
            if np.random.rand() > 0.5:
                img  = np.ascontiguousarray(img[:,  ::-1])
                mask = np.ascontiguousarray(mask[:, ::-1])
            # v-flip
            if np.random.rand() > 0.5:
                img  = np.ascontiguousarray(img[::-1,  :])
                mask = np.ascontiguousarray(mask[::-1, :])
            # intensity (image only)
            if np.random.rand() > 0.5:
                img = np.clip(img + np.random.uniform(-0.05, 0.05), 0, 1).astype(np.float32)
            if np.random.rand() > 0.7:
                img = np.clip(img + np.random.randn(*img.shape).astype(np.float32) * 0.02, 0, 1)

        img  = torch.from_numpy(img[None]).float()    # (1,H,W)
        mask = torch.from_numpy(mask[None]).float()   # (1,H,W)
        return img, mask, label

train_set = BUSILesionDataset(os.path.join(PROCESSED_DIR, 'train'), augment=True,  mask_source='pred')
val_set   = BUSILesionDataset(os.path.join(PROCESSED_DIR, 'val'),   augment=False, mask_source='pred')
test_set  = BUSILesionDataset(os.path.join(PROCESSED_DIR, 'test'),  augment=False, mask_source='pred')

print(f'train={len(train_set)} val={len(val_set)} test={len(test_set)}')
print(f'train binary distrib: 0={train_set.binary_labels.count(0)}  1={train_set.binary_labels.count(1)}')

In [ ]:
# --- Soft-guided attention (kept verbatim from integrated_v3, light comments) ---
class SoftGuidedAttention(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 4, 1, bias=False),
            nn.BatchNorm2d(in_channels // 4),
            nn.ReLU(inplace=False),
            nn.Conv2d(in_channels // 4, 1, 1, bias=False),
            nn.Sigmoid(),
        )
        # Learnable mask-influence weight: alpha = sigmoid(alpha_raw) in [0,1]
        self.alpha_raw = nn.Parameter(torch.tensor(0.0))

    def forward(self, x, mask=None):
        spatial_att = self.spatial_conv(x)              # (B,1,H',W')
        if mask is not None:
            mask_d = F.interpolate(mask, size=spatial_att.shape[2:],
                                   mode='bilinear', align_corners=False)
            alpha  = torch.sigmoid(self.alpha_raw)
            # Soft gate: inside lesion -> ~1, outside -> (1-alpha) > 0 (info preserved)
            spatial_att = spatial_att * (alpha * mask_d + (1.0 - alpha))
        return x * spatial_att

class EfficientNetLesionAttn(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4, pretrained=True):
        super().__init__()
        weights = tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = tv_models.efficientnet_b0(weights=weights)
        # 3-ch -> 1-ch first conv, averaging pretrained weights
        old = backbone.features[0][0]
        new = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        if pretrained:
            new.weight.data = old.weight.data.mean(dim=1, keepdim=True)
        backbone.features[0][0] = new
        self.features = backbone.features
        self.attention = SoftGuidedAttention(in_channels=1280)
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(1280, 256), nn.ReLU(inplace=False),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes),
        )

    def forward(self, x, mask=None):
        return self.classifier(self.attention(self.features(x), mask))

    def freeze_bn(self):
        """Set all BatchNorm layers to eval mode (use ImageNet running stats)."""
        for m in self.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
                for p in m.parameters():
                    p.requires_grad = False

model = EfficientNetLesionAttn(num_classes=2, pretrained=True).to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# --- Sampler, optimizer, scheduler, loss ---
BATCH_CLF      = 8
EPOCHS_CLF     = 25
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
PATIENCE       = 10
NUM_WORKERS    = 2

# Weighted random sampler: sample each class with equal probability
class_count = np.bincount(train_set.binary_labels)
weights_per_sample = np.array([1.0 / class_count[y] for y in train_set.binary_labels])
sampler = WeightedRandomSampler(weights_per_sample, num_samples=len(train_set), replacement=True)

train_loader = DataLoader(train_set, batch_size=BATCH_CLF, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_CLF, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_CLF, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

loss_fn   = nn.CrossEntropyLoss()  # sampler handles balance, no class weights needed
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_CLF, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda')

print(f'Class count (binary): {class_count}')
print(f'Steps per epoch     : {len(train_loader)}')

In [ ]:
# --- Training loop ---
train_losses, val_losses, val_f1s, val_accs, alphas = [], [], [], [], []
best_val_f1, patience_count = -1, 0

for epoch in range(EPOCHS_CLF):
    # ---- Train ----
    model.train()
    model.freeze_bn()  # IMPORTANT: small batch -> use ImageNet BN stats

    epoch_loss, n_steps = 0.0, 0
    for img, mask, lbl in train_loader:
        img  = img.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        lbl  = lbl.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out  = model(img, mask)
            loss = loss_fn(out, lbl)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item(); n_steps += 1
    scheduler.step()
    train_losses.append(epoch_loss / n_steps)

    # ---- Validate ----
    model.eval()
    vl_sum, all_pred, all_true = 0.0, [], []
    with torch.no_grad():
        for img, mask, lbl in val_loader:
            img, mask, lbl = img.to(device), mask.to(device), lbl.to(device)
            with torch.amp.autocast('cuda'):
                out = model(img, mask)
                vl_sum += loss_fn(out, lbl).item()
            all_pred.extend(out.argmax(1).cpu().numpy())
            all_true.extend(lbl.cpu().numpy())
    val_loss = vl_sum / len(val_loader)
    val_acc  = accuracy_score(all_true, all_pred)
    val_f1   = f1_score(all_true, all_pred, average='macro')
    val_losses.append(val_loss); val_accs.append(val_acc); val_f1s.append(val_f1)

    alpha_v = torch.sigmoid(model.attention.alpha_raw).item()
    alphas.append(alpha_v)
    print(f'Epoch {epoch+1:02d}/{EPOCHS_CLF} | '
          f'train_loss={train_losses[-1]:.4f} | val_loss={val_loss:.4f} | '
          f'val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | alpha={alpha_v:.3f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), CLF_WEIGHTS)
        print(f'  ✓ best val F1 -> saved')
        patience_count = 0
    else:
        patience_count += 1
    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1} (no F1 improvement for {PATIENCE} epochs)')
        break

print(f'\nBest val macro-F1: {best_val_f1:.4f}')

In [ ]:
# --- Training curves ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
xs = range(1, len(train_losses)+1)
axes[0].plot(xs, train_losses, 'o-', label='train')
axes[0].plot(xs, val_losses,   'o-', label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(xs, val_accs, 'o-', label='val acc')
axes[1].plot(xs, val_f1s,  'o-', label='val macro-F1')
axes[1].set_title('Validation metrics'); axes[1].set_xlabel('epoch'); axes[1].legend()

axes[2].plot(xs, alphas, 'o-', color='purple')
axes[2].set_title('Soft-gate α (mask influence)'); axes[2].set_xlabel('epoch')
axes[2].set_ylim(0, 1); axes[2].axhline(0.5, color='gray', ls='--', alpha=0.5)
plt.tight_layout(); plt.show()

---
## 5. Test-Set Evaluation

In [ ]:
# Reload best weights
model.load_state_dict(torch.load(CLF_WEIGHTS, map_location=device))
model.eval()

all_pred, all_prob, all_true = [], [], []
with torch.no_grad():
    for img, mask, lbl in test_loader:
        img, mask = img.to(device), mask.to(device)
        out = model(img, mask)
        prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
        pred = out.argmax(1).cpu().numpy()
        all_prob.extend(prob); all_pred.extend(pred); all_true.extend(lbl.numpy())

all_pred = np.array(all_pred); all_prob = np.array(all_prob); all_true = np.array(all_true)

print('=== Test set results ===')
print(f'Accuracy : {accuracy_score(all_true, all_pred):.4f}')
print(f'Macro-F1 : {f1_score(all_true, all_pred, average="macro"):.4f}')
try:
    print(f'AUC      : {roc_auc_score(all_true, all_prob):.4f}')
except ValueError:
    print('AUC      : (only one class in test set?)')
print()
print(classification_report(all_true, all_pred, target_names=BINARY_NAMES, digits=4))

In [ ]:
# --- Confusion matrix + ROC ---
import seaborn as sns
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(all_true, all_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=BINARY_NAMES, yticklabels=BINARY_NAMES, ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (Test)')

fpr, tpr, _ = roc_curve(all_true, all_prob)
auc_val = roc_auc_score(all_true, all_prob)
axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc_val:.4f}')
axes[1].plot([0,1], [0,1], 'k--', alpha=0.4)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve (Test)'); axes[1].legend()
plt.tight_layout(); plt.show()

---
## 6. (Optional) Ablation — predicted mask vs. GT mask

Re-evaluate the SAME trained classifier feeding it the GT mask instead of
U-Net's predicted mask. Gap measures how much performance is bottlenecked by
U-Net errors. Small gap = robust pipeline; large gap = invest more in U-Net.


In [ ]:
gt_test_set = BUSILesionDataset(os.path.join(PROCESSED_DIR, 'test'),
                                augment=False, mask_source='gt')
gt_loader   = DataLoader(gt_test_set, batch_size=BATCH_CLF, shuffle=False, num_workers=NUM_WORKERS)

model.eval()
gt_pred, gt_prob, gt_true = [], [], []
with torch.no_grad():
    for img, mask, lbl in gt_loader:
        img, mask = img.to(device), mask.to(device)
        out = model(img, mask)
        prob = F.softmax(out, dim=1)[:, 1].cpu().numpy()
        gt_pred.extend(out.argmax(1).cpu().numpy())
        gt_prob.extend(prob); gt_true.extend(lbl.numpy())

print(f'                  Accuracy   Macro-F1   AUC')
print(f'Pred mask (test): {accuracy_score(all_true, all_pred):.4f}     '
      f'{f1_score(all_true, all_pred, average="macro"):.4f}    '
      f'{roc_auc_score(all_true, all_prob):.4f}')
print(f'GT   mask (test): {accuracy_score(gt_true, gt_pred):.4f}     '
      f'{f1_score(gt_true, gt_pred, average="macro"):.4f}    '
      f'{roc_auc_score(gt_true, gt_prob):.4f}')

---
### Notes for the next iteration

- If `α` ends up close to 0, the classifier is ignoring the mask — investigate
  U-Net quality or attention placement.
- If GT-mask oracle is much better than predicted-mask, U-Net is the bottleneck
  → consider larger U-Net, longer training, or pretrained encoder.
- If both are close to baseline (no mask), the soft-attention module placement
  may be wrong — try injecting attention at multiple feature stages.
